# Categorize Responses in the Behavioural Set

In [ ]:
import sys, os
from pathlib import Path 

PROJECT_ROOT = Path("/Users/robertagarcia/Desktop/learning/bert_symptom_ner")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

sys.path

import torch
import json
from dataclasses import dataclass
from pydantic import BaseModel, Field
from transformers import AutoTokenizer, AutoModelForTokenClassification

# Local imports
from config import settings
from gcp_utils import download_from_gcs
from inference.v01.inference_utils import predict_word_level, word_labels_to_spans
from error_analysis.error_categorization import ErrorCategorizer,  _run_through_behavioural_set


_run_through_behavioural_set
from error_analysis.error_taxonomy import BehaviouralExample

VERSION = os.environ["VERSION"]
MODEL_NAME = "dmis-lab/biobert-base-cased-v1.1" 
RUN_IDX = 0 # manually checked this in GCP bucket
INFERENCE_PIPELINE_VERSION = "v01"


# 2) Load label mappings from the local JSON (same labels across all versions)
with open(f"../{VERSION}/data/id2label.json", "r") as f:
    id2label = {int(k): v for k, v in json.load(f).items()}

label2id = {v: k for k, v in id2label.items()}

print(f"✅ Loaded {len(id2label)} labels: {id2label}")

# Move model to device
if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"
device

# Load the Model 

In [ ]:

LOCAL_MODEL_DIR = f"{PROJECT_ROOT}/{VERSION}/downloaded_models/dmis-lab/biobert-base-cased-v1.1/run_{RUN_IDX}"

if not os.path.isdir(LOCAL_MODEL_DIR):
    #if nor any(os.scandir(LOCAL_MODEL_DIR)):
    print("Downloading from GCP")
    # Download model from GCP: 
    download_from_gcs(gcs_path="v04/runs/dmis-lab/biobert-base-cased-v1.1/run_0", local_path=LOCAL_MODEL_DIR, bucket_name="ner_training_data_results")

tokenizer = AutoTokenizer.from_pretrained(LOCAL_MODEL_DIR)
model = AutoModelForTokenClassification.from_pretrained(LOCAL_MODEL_DIR)
model = model.to(device)
model.eval()
print(f"✅ Loaded model and tokenizer from {LOCAL_MODEL_DIR}")


# Load the Behavioural Set

In [ ]:
# Load behavioural evaluation set
print("\n📂 Loading behavioural evaluation set...")
with open(f"{PROJECT_ROOT}/behavioural_set.json", "r") as f:
    behavioural_set = json.load(f)


# Fit the behavioural set into the dataclasses
for k,examples in behavioural_set.items():
    for i,ex in enumerate(examples):
        behavioural_set[k][i] = BehaviouralExample(**ex)



# Instantiate the Error Categorizer

In [ ]:
error_categorizer = ErrorCategorizer()

# Run Error Categorization thoughout the entire dataset

In [ ]:
results = _run_through_behavioural_set(
    behavioural_set=behavioural_set,
    error_categorizer=error_categorizer,
    model=model,
    id2label=id2label,
    tokenizer=tokenizer,
    device=device
)

# These results are very important they dictate the next steps for improving the model!
results["error_counts"]